# EcoHome Energy Advisor - RAG Setup

In this notebook, you'll set up the Retrieval-Augmented Generation (RAG) pipeline for the EcoHome Energy Advisor. This will allow the agent to access and cite relevant energy-saving tips and best practices.

## Learning Objectives
- Set up ChromaDB vector store
- Load and process energy-saving documents
- Create embeddings for document chunks
- Implement semantic search functionality
- Test the RAG pipeline

## Documents Available
- `tip_device_best_practices.txt` - Device-specific optimization tips
- `tip_energy_savings.txt` - General energy-saving strategies


## 1. Import Required Libraries


In [1]:
# Import the necessary libraries for RAG setup
import os
from langchain_chroma  import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

## 2. Load and Process Documents


In [3]:
# Load the energy-saving tip documents
# Load both tip_device_best_practices.txt and tip_energy_savings.txt
# Use TextLoader to load the documents

# Load all energy-saving tip documents from the documents directory
documents = []
doc_dir = "data/documents"

for filename in os.listdir(doc_dir):
    if filename.endswith(".txt"):
        doc_path = os.path.join(doc_dir, filename)
        loader = TextLoader(doc_path, encoding="utf-8")
        docs = loader.load()
        documents.extend(docs)
        print(f"Loaded {len(docs)} document(s) from {doc_path}")

print(f"\nTotal documents loaded: {len(documents)}")

Loaded 1 document(s) from data/documents\tip_device_best_practices.txt
Loaded 1 document(s) from data/documents\tip_energy_savings.txt
Loaded 1 document(s) from data/documents\tip_energy_storage.txt
Loaded 1 document(s) from data/documents\tip_hvac_optimization.txt
Loaded 1 document(s) from data/documents\tip_renewable_energy.txt
Loaded 1 document(s) from data/documents\tip_seasonal_management.txt
Loaded 1 document(s) from data/documents\tip_smart_home_automation.txt

Total documents loaded: 7


## 3. Split Documents into Chunks


In [4]:
# Split documents into smaller chunks for better retrieval
# Use RecursiveCharacterTextSplitter with appropriate chunk_size and chunk_overlap
# Experiment with different chunk sizes (e.g., 500, 1000, 1500 characters)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)

# Split the documents
splits = text_splitter.split_documents(documents)
print(f"Split {len(documents)} documents into {len(splits)} chunks")

# Show sample chunk
if splits:
    print(f"\nSample chunk (first 200 characters):")
    print(splits[0].page_content[:200] + "...")


Split 7 documents into 21 chunks

Sample chunk (first 200 characters):
Large devices like electric vehicles, washing machines and dishwashers often support delayed start or timer functions. Schedule these devices to run outside of peak electricity pricing hours or during...


## 4. Create Vector Store


In [5]:
import shutil

# Set up the persist directory — wipe first to avoid duplicate entries on rebuild
persist_directory = "data/vectorstore"
shutil.rmtree(persist_directory, ignore_errors=True)
os.makedirs(persist_directory)

# Initialize embeddings
embeddings = OpenAIEmbeddings(
    api_key=os.getenv("OPENAI_API_KEY")
)

# Create the vector store
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    persist_directory=persist_directory
)

print(f"Vector store created and persisted to {persist_directory}")
print(f"Total vectors stored: {len(splits)}")

Vector store created and persisted to data/vectorstore
Total vectors stored: 21


## 5. Test the RAG Pipeline


In [6]:
# Test the search functionality
# Try different queries related to energy optimization
# Test queries like:
# - "electric vehicle charging tips"
# - "thermostat optimization"
# - "dishwasher energy saving"
# - "solar power maximization"

test_queries = [
    "electric vehicle charging tips",
    "thermostat optimization",
    "dishwasher energy saving",
    "solar power maximization",
    "HVAC system efficiency",
    "pool pump scheduling"
]

print("=== Testing Vector Search ===")
for query in test_queries:
    print(f"\nQuery: '{query}'")
    docs = vectorstore.similarity_search(query, k=2)
    for i, doc in enumerate(docs):
        print(f"  Result {i+1}: {doc.page_content[:100]}...")


=== Testing Vector Search ===

Query: 'electric vehicle charging tips'
  Result 1: 3. Smart Lighting
- Replace all bulbs with LED; LEDs use 75% less energy than incandescent bulbs
- U...
  Result 2: Large devices like electric vehicles, washing machines and dishwashers often support delayed start o...

Query: 'thermostat optimization'
  Result 1: HVAC Optimization Strategies for Smart Homes

1. Thermostat Scheduling
- Set thermostat to 68°F (20°...
  Result 2: Use power strips to easily turn off multiple devices at once. Many electronics continue to draw powe...

Query: 'dishwasher energy saving'
  Result 1: Dishwasher Best Practices:
- Only run when completely full
- Use the energy-saving or eco mode when ...
  Result 2: Large devices like electric vehicles, washing machines and dishwashers often support delayed start o...

Query: 'solar power maximization'
  Result 1: Renewable Energy Integration for Smart Homes

1. Solar Panel Optimization
- Maximize solar self-cons...
  Result 2: 4

## 6. Test the Search Tool


In [7]:
# # Test the search_energy_tips tool from tools.py
# # Import and test the tool with various queries
# # Verify that it returns relevant results

# from tools import search_energy_tips

# # Test the search_energy_tips function
# print("=== Testing search_energy_tips Tool ===")

# test_queries = [
#     "electric vehicle charging",
#     "thermostat settings",
#     "dishwasher optimization",
#     "solar power tips"
# ]

# for query in test_queries:
#     print(f"\nQuery: '{query}'")
#     result = search_energy_tips.invoke(
#         input={
#             "query": query, 
#             "max_results": 3,
#         }
#     )
    
#     if "error" in result:
#         print(f"  Error: {result['error']}")
#     else:
#         print(f"  Found {result['total_results']} results")
#         for i, tip in enumerate(result['tips']):
#             print(f"    {i+1}. {tip['content'][:100]}...")
#             print(f"       Source: {tip['source']}")
#             print(f"       Relevance: {tip['relevance_score']}")


In [ ]:
from tools import search_energy_tips

print("=== Testing search_energy_tips Tool ===")

test_queries = [
    "electric vehicle charging",
    "thermostat settings",
    "dishwasher optimization",
    "solar power tips",
    "HVAC pre-cooling peak hours",
    "battery storage charging strategy",
    "smart home automation scheduling",
    "seasonal energy management winter",
    "renewable energy solar self-consumption"
]

for query in test_queries:
    print(f"\nQuery: '{query}'")
    result = search_energy_tips.invoke(input={"query": query, "max_results": 3})

    if "error" in result:
        print(f"  Error: {result['error']}")
    else:
        print(f"  Found {result['total_results']} results")
        for tip in result['tips']:
            print(f"    Result {tip['rank']}: {tip['content'][:100]}...")
            print(f"      Source: {tip['source']}")
            print(f"      Relevance: {tip['relevance_score']}")

In [9]:
# import shutil
# shutil.rmtree("data/vectorstore", ignore_errors=True)